# NB2 — Small-File Problem & OPTIMIZE + Z-order (lightweight)

Maps to slide §6 Storage Optimization (& Anti-Patterns) + deliverable bullet 2.

> Spark equivalent: `OPTIMIZE delta.\`path\` ZORDER BY (user_id)`
> delta-rs:        `dt.optimize.compact()` + `dt.optimize.z_order(["user_id"])`

**Key idea:** Z-order's value is *file-skipping*. With min/max stats per
file, Delta prunes files whose range can't contain your predicate. That
only works when the table has *multiple files* — if `compact()` collapses
everything to one file, Z-order has nothing to prune.

In [1]:
import _setup  # noqa: F401  -- adds scripts/ to sys.path
import time, random
import polars as pl
import duckdb
from deltalake import DeltaTable, write_deltalake
from lakehouse import path, reset

table_path = path("scratch", "events_smallfiles")
reset(table_path)  # idempotent

## 1. Manufacture the small-file problem

200 tiny appends → 200 small files. Realistic streaming-ingestion shape.
Each batch is 5K rows × wider schema (≈200 B/row payload) so post-compaction
we still have multiple files with a 256 KB target — required for Z-order
skipping to actually skip something in the benchmark.

In [2]:
random.seed(42)
TARGET_USER = 4242  # the point-query needle

# Pre-built payload pool keeps row generation fast while making each row fat
# enough that compaction doesn't collapse 1M rows into a single 8 MB file.
PAYLOADS = [("p" * 200) + str(i) for i in range(64)]

for batch in range(200):
    rows = pl.DataFrame({
        "event_id":  list(range(batch * 5_000, (batch + 1) * 5_000)),
        "kind":      [random.choice(["click", "view", "scroll", "purchase"]) for _ in range(5_000)],
        # 100K distinct users → before z-order, the target user is scattered
        # across many files; after z-order it clusters into ~1 file.
        "user_id":   [random.randint(1, 100_000) for _ in range(5_000)],
        "payload":   [random.choice(PAYLOADS) for _ in range(5_000)],
    })
    mode = "overwrite" if batch == 0 else "append"
    write_deltalake(table_path, rows.to_arrow(), mode=mode)

dt = DeltaTable(table_path)
files_before = len(dt.file_uris())
print(f"Files before OPTIMIZE: {files_before}")

Files before OPTIMIZE: 200


### 📝 Nhận xét — Tái hiện small-file problem

- 200 lần `append`, mỗi lần 5.000 dòng → **200 file Parquet** (~66 KB/file, tổng ~13,2 MB, 1 triệu dòng) và 200 commit trong `_delta_log/`. Đạt ngưỡng ≥ 100 file của rubric.
- Đây đúng là hình dạng của **streaming ingestion**: mỗi micro-batch ghi một file nhỏ. Không có code nào sai — vấn đề sinh ra từ việc ghi thường xuyên mà không có job bảo trì.
- Mỗi file mang **chi phí cố định** khi đọc: mở/liệt kê file (trên S3 là một HTTP request), đọc footer Parquet, và thêm một entry phải replay trong log. Nhiều file nhỏ ⇒ phần lớn thời gian dành cho overhead thay vì đọc dữ liệu.


## 2. Benchmark BEFORE optimize

We use delta-rs's native filter pushdown (`to_pyarrow_table(filters=...)`)
which reads per-file `min`/`max` stats from the transaction log and skips
files whose range can't contain the predicate. That's the mechanism we
want to measure — the same one Spark/Trino use.

In [3]:
def bench(label: str, runs: int = 3) -> float:
    """Median of `runs` point-queries using Delta's stats-based file pruning."""
    times = []
    n = 0
    for _ in range(runs):
        dt_local = DeltaTable(table_path)  # fresh metadata read
        t0 = time.perf_counter()
        tbl = dt_local.to_pyarrow_table(
            filters=[("user_id", "=", TARGET_USER), ("kind", "=", "purchase")]
        )
        n = tbl.num_rows
        times.append(time.perf_counter() - t0)
    times.sort()
    median = times[len(times) // 2]
    print(f"{label:25s}  count={n}  median={median*1000:6.1f} ms  (n={runs})")
    return median

before = bench("BEFORE OPTIMIZE")

BEFORE OPTIMIZE            count=5  median= 311.3 ms  (n=3)


### 📝 Nhận xét — Benchmark trước OPTIMIZE

- Truy vấn điểm `user_id = 4242 AND kind = 'purchase'` trả `count=5`, median **311,3 ms**.
- `to_pyarrow_table(filters=...)` dùng **predicate pushdown**: delta-rs so điều kiện lọc với `minValues`/`maxValues` của từng file trong log để bỏ qua file không thể chứa kết quả.
- Nhưng ở đây cơ chế đó **vô dụng**: dữ liệu đến theo thời gian, mỗi file là một mẫu ngẫu nhiên của `user_id` 1–100.000 nên khoảng min/max của mọi file ≈ `[1, 100000]` (xem bảng so sánh version bên dưới: 200/200 file "có thể chứa" 4242). Engine buộc phải mở **cả 200 file** để tìm 5 dòng.


## 3. OPTIMIZE (compact small files) + Z-ORDER (co-locate by user_id)

`target_size` is 256 KB so we keep multiple files post-compact — enough
for Z-order's file-skipping to actually skip something.

In [4]:
TARGET_SIZE = 256 * 1024  # 256 KB — keeps ~50 files post-compact for visible pruning

dt = DeltaTable(table_path)
dt.optimize.compact(target_size=TARGET_SIZE)
dt.optimize.z_order(["user_id"], target_size=TARGET_SIZE)

dt = DeltaTable(table_path)  # refresh
files_after = len(dt.file_uris())
print(f"Files after OPTIMIZE+ZORDER: {files_after}  (was {files_before})")

Files after OPTIMIZE+ZORDER: 55  (was 200)


### 📝 Nhận xét — OPTIMIZE: compact + Z-order

- Hai thao tác, hai mục đích khác nhau, mỗi thao tác là **một commit mới** (`operation: OPTIMIZE`):
  - `compact()` — **bin-packing**: gom file nhỏ thành file gần `target_size` → 200 → 67 file. Không sắp xếp dữ liệu.
  - `z_order(["user_id"])` — **sắp xếp lại** dữ liệu theo đường cong Z để các `user_id` gần nhau nằm cùng file → 67 → 55 file với khoảng `user_id` hẹp, không chồng lấn.
- `target_size = 256 KB` được chọn **cố ý nhỏ** (production thường 128 MB–1 GB): nếu gộp 1 triệu dòng thành 1 file thì không còn file nào để bỏ qua. Pruning cần **nhiều file, mỗi file hẹp**.
- Z-order thay vì partition theo `user_id`: cột có 100.000 giá trị → partition sẽ tạo 100.000 thư mục, tức tái tạo chính small-file problem. Z-order phù hợp cột **high-cardinality**.


## 4. Benchmark AFTER

In [5]:
after = bench("AFTER OPTIMIZE+ZORDER")
print(f"\nSpeedup: {before/max(after, 1e-6):.1f}×  (target ≥ 3×)")
print(f"File reduction: {files_before} → {files_after}  ({files_before/max(files_after,1):.0f}× fewer)")

AFTER OPTIMIZE+ZORDER      count=5  median=  30.3 ms  (n=3)

Speedup: 10.3×  (target ≥ 3×)
File reduction: 200 → 55  (4× fewer)


### 📝 Nhận xét — Benchmark sau OPTIMIZE

- Cùng truy vấn, **cùng kết quả `count=5`** → OPTIMIZE chỉ đổi cách bố trí vật lý, không đổi dữ liệu logic (commit OPTIMIZE ghi `dataChange: false`).
- Median **311,3 ms → 30,3 ms ⇒ speedup 10,3×** (≥ 3× ✅).
- Wall-clock **dao động theo máy và lần chạy**: lần chạy thử trước đó trên cùng máy đo được 613,7 → 41,4 ms (14,8×). Nguyên nhân: cache của hệ điều hành, tải CPU/SSD tại thời điểm đo, chỉ lấy median của 3 lần. Vì vậy speedup không phải bằng chứng ổn định — thước đo tất định là **số file bị bỏ qua** (cell bên dưới).
- File reduction chỉ 200 → 55 (~4×) vì `target_size` nhỏ; mục tiêu của notebook là pruning, không phải gộp tối đa.


## 5. Why this works — inspect file-level stats

Delta stores per-file `min`/`max` for stat-eligible columns in the
transaction log. After Z-order, `user_id` ranges per file are tight and
non-overlapping; the engine prunes files whose range excludes 4242.

In [9]:
import json
import os

log_dir = os.path.join(table_path, "_delta_log")
last_log = sorted(f for f in os.listdir(log_dir) if f.endswith(".json"))[-1]
print(f"Inspecting {last_log}:")
hits = 0
ranges = []
with open(os.path.join(log_dir, last_log)) as fh:
    for line in fh:
        e = json.loads(line)
        if "add" in e and "stats" in e["add"]:
            stats = json.loads(e["add"]["stats"])
            mn = stats.get("minValues", {}).get("user_id")
            mx = stats.get("maxValues", {}).get("user_id")
            if mn is not None:
                ranges.append((mn, mx))
                if mn <= TARGET_USER <= mx:
                    hits += 1
for mn, mx in sorted(ranges):
    marker = " ← contains target" if mn <= TARGET_USER <= mx else ""
    print(f"  file user_id range: [{mn:>6}, {mx:>6}]{marker}")

# Slide-5 deliverable allows EITHER metric — print both so the student can
# pick whichever the grader screenshots:
#   Speedup ≥ 3×              (wall-clock, noisy on local SSD)
#   Files-pruned ratio ≥ 10×  (deterministic, the truer Z-order metric)
pruned_ratio = files_after / max(hits, 1)
print(
    f"\n──── Z-order deliverable metrics ────\n"
    f"  Speedup (wall-clock):   {before/max(after, 1e-6):>5.1f}×   (target ≥ 3×)\n"
    f"  Files-pruned ratio:     {pruned_ratio:>5.1f}×   (target ≥ 10×)   "
    f"[{hits} of {files_after} files cover user_id={TARGET_USER}]"
)

Inspecting 00000000000000000201.json:
  file user_id range: [     1,   1851]
  file user_id range: [  1851,   3696]
  file user_id range: [  3696,   5534] ← contains target
  file user_id range: [  5535,   7366]
  file user_id range: [  7366,   9178]
  file user_id range: [  9178,  11014]
  file user_id range: [ 11014,  12873]
  file user_id range: [ 12874,  14722]
  file user_id range: [ 14722,  16568]
  file user_id range: [ 16568,  18406]
  file user_id range: [ 18406,  20263]
  file user_id range: [ 20263,  22115]
  file user_id range: [ 22115,  23954]
  file user_id range: [ 23954,  25789]
  file user_id range: [ 25790,  27641]
  file user_id range: [ 27641,  29487]
  file user_id range: [ 29487,  31315]
  file user_id range: [ 31315,  33149]
  file user_id range: [ 33149,  34985]
  file user_id range: [ 34985,  36811]
  file user_id range: [ 36811,  38666]
  file user_id range: [ 38666,  40498]
  file user_id range: [ 40498,  42319]
  file user_id range: [ 42319,  44175]
  file u

### 📝 Nhận xét — File-level stats sau Z-order

- Commit cuối (`…201.json`) chứa 55 action `add`, mỗi file có khoảng `user_id` rộng ~1.800 giá trị, **nối tiếp và không chồng lấn**: `[1, 1851]`, `[1851, 3696]`, `[3696, 5534]`, …
- Chỉ **1/55 file** (`[3696, 5534]`) có khoảng chứa 4242 ⇒ **files-pruned ratio = 55×** (≥ 10× ✅). 54 file bị loại chỉ bằng cách đọc stats trong `_delta_log`, không cần mở file.
- So với trước tối ưu: số file phải đọc giảm từ **200 → 1**. Đây là thước đo tất định (tính từ metadata) nên lặp lại giống nhau mỗi lần chạy, khác với speedup.
- Đây chính là `stats` trong action `add` đã thấy ở NB1 — Z-order không thêm cơ chế mới mà làm cho stats sẵn có **trở nên hữu ích**.


In [10]:
# Tách riêng tác dụng của compact() và z_order(): so 3 version cuối của bảng
import glob
import pyarrow as pa

latest = DeltaTable(table_path).version()
steps = [(latest - 2, "trước OPTIMIZE"), (latest - 1, "sau compact()"), (latest, "sau z_order()")]
print(f"{'version':>8} {'bước':16} {'files':>5} {'chứa 4242':>10} {'độ rộng min-max TB':>19} {'MB':>6}")
for v, label in steps:
    a = pa.table(DeltaTable(table_path, version=v).get_add_actions(flatten=True)).to_pydict()
    mn, mx = a["min.user_id"], a["max.user_id"]
    hits_v = sum(x <= TARGET_USER <= y for x, y in zip(mn, mx))
    width = sum(y - x for x, y in zip(mn, mx)) / len(mn)
    print(f"{'v' + str(v):>8} {label:16} {len(mn):>5} {hits_v:>10} {width:>19,.0f} {sum(a['size_bytes'])/1e6:>6.2f}")

op = DeltaTable(table_path).history(1)[0]
print(f"\nCommit cuối: operation={op['operation']}, "
      f"strategy={op['operationMetrics'].get('plannerStrategy')}")
on_disk = len(glob.glob(f"{table_path}/*.parquet"))
print(f"File Parquet trên đĩa: {on_disk}  |  file bảng đang tham chiếu: {len(DeltaTable(table_path).file_uris())}")

 version bước             files  chứa 4242  độ rộng min-max TB     MB
    v199 trước OPTIMIZE     200        200              99,960  13.19
    v200 sau compact()       67         67              99,987   8.17
    v201 sau z_order()       55          1               1,818   6.77

Commit cuối: operation=OPTIMIZE, strategy=zOrder
File Parquet trên đĩa: 322  |  file bảng đang tham chiếu: 55


### 📝 Nhận xét — Tách riêng compact và Z-order

| Version | Bước | Files | File chứa 4242 | Độ rộng min–max TB | MB |
|---|---|---:|---:|---:|---:|
| v199 | trước OPTIMIZE | 200 | 200 | ~99.960 | 13,19 |
| v200 | sau `compact()` | 67 | **67** | ~99.987 | 8,17 |
| v201 | sau `z_order()` | 55 | **1** | **~1.818** | 6,77 |

- **Compaction một mình không tạo ra pruning**: sau `compact()` vẫn phải đọc 67/67 file vì mỗi file vẫn phủ gần toàn bộ dải `user_id`. Compaction giảm **chi phí mỗi file** (ít lần mở file, ít footer, log ngắn hơn).
- **Z-order mới tạo ra pruning**: độ rộng khoảng min–max giảm ~55 lần (99.960 → 1.818) → 54/55 file bị loại. Z-order giảm **số file phải đọc**.
- Dung lượng giảm 13,19 → 6,77 MB: file lớn hơn và đã sắp xếp nén/dictionary-encode tốt hơn, ít footer/metadata hơn.
- **322 file Parquet trên đĩa nhưng bảng chỉ tham chiếu 55**: OPTIMIZE chỉ đánh dấu file cũ là `remove` (tombstone) trong log, không xoá vật lý. 267 file cũ vẫn tốn dung lượng — nhưng cũng nhờ đó vẫn time travel được về v199 (NB3). Thu hồi chúng là việc của `VACUUM` (NB6).


## ✅ Deliverable check
- [ ] Speedup ≥ 3× **or** files-pruned ratio ≥ 10× (slide §6 allows either)
- [ ] File count dropped substantially after compact()
- [ ] Stats inspection shows ~1 file covers `user_id=4242`
- [ ] Screenshot the printed numbers

In [8]:
speedup = before / max(after, 1e-6)
checks = {
    "compaction reduced file count":  files_after < files_before,
    "speedup ≥ 3x OR pruning ≥ 10x":  speedup >= 3 or pruned_ratio >= 10,
    "stats isolate the target user":  hits <= max(2, files_after // 4),
}
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
print(f"\n  (speedup={speedup:.1f}x, pruning={pruned_ratio:.1f}x — the slide accepts EITHER;")
print("   wall-clock is noisy on a laptop, which is why file-pruning is the fallback.)")
assert all(checks.values()), "NB2 incomplete — see FAIL rows above"
print("\nNB2 complete.")

  [PASS] compaction reduced file count
  [PASS] speedup ≥ 3x OR pruning ≥ 10x
  [PASS] stats isolate the target user

  (speedup=10.3x, pruning=55.0x — the slide accepts EITHER;
   wall-clock is noisy on a laptop, which is why file-pruning is the fallback.)

NB2 complete.


### 🎯 Kết luận NB2

Small-file problem sinh ra tự nhiên từ streaming ingestion (200 file nhỏ). **Compaction** giảm chi phí trên mỗi file nhưng không giúp bỏ qua file; **Z-order** làm khoảng min/max của từng file hẹp lại để stats trong `_delta_log` loại được 54/55 file — từ 200 file phải đọc xuống 1. Kết quả: pruning **55×** (tất định) và speedup **10,3×** (dao động theo máy). OPTIMIZE là transaction `dataChange: false`, giữ nguyên dữ liệu logic, và để lại file cũ cho tới khi VACUUM — tiền đề của NB3 (time travel) và NB6 (maintenance).
